# **Complete repository step-by-step diagnostics.**

In [1]:
# try-catch wrapper for this file

def validate_call(label, func, *args, **kwargs):
    try:
        result = func(*args, **kwargs)
    except Exception as e:
        print(f"INVALID: {label} - {e}")
        return None
    else:
        print(f"VALID: {label}")
        return result

## **Node** (in node.py)

In [7]:
from utils.node import Node

# Not much to analyze. We just need to know it works.

test_node_configs = [
    (1, 2, 0),
    (2, 1, 0),
    (1, 2, 1),
    (2, 1, 1),
    (1, 2, 2),
    (2, 1, 2),
    (1, 2, 3),
    (2, 1, 3),
    (1, 2, 4),
    (2, 1, -1),
]

for lon, lat, layer in test_node_configs:
    node = validate_call(
        "Node(124.2, 8.2, 1)",
        Node,
        124.2,
        8.2,
        1
    )

VALID: Node(124.2, 8.2, 1)
VALID: Node(124.2, 8.2, 1)
VALID: Node(124.2, 8.2, 1)
VALID: Node(124.2, 8.2, 1)
VALID: Node(124.2, 8.2, 1)
VALID: Node(124.2, 8.2, 1)
VALID: Node(124.2, 8.2, 1)
VALID: Node(124.2, 8.2, 1)
VALID: Node(124.2, 8.2, 1)
VALID: Node(124.2, 8.2, 1)


## **DirEdge** (in directed_edge.py)

In [17]:

from utils.node import Node
from utils.directed_edge import DirEdge, _stitch

# sample nodes:

n1l1 = Node(8.2393, 124.2384, 1) # msu-iit
n2l1 = Node(8.237768, 124.243862, 1) # 7-11 msu-iit
n3l1 = Node(8.239, 124.238, 1) # mcdonalds tibanga

n1l2 = Node(8.2393, 124.2384, 2)
n2l2 = Node(8.237768, 124.243862, 2)
n3l2 = Node(8.239, 124.238, 2)

n1l3 = Node(8.2393, 124.2384, 3)
n2l3 = Node(8.237768, 124.243862, 3)
n3l3 = Node(8.239, 124.238, 3)

n1l0 = Node(8.2393, 124.2384, 0)
n2l0 = Node(8.237768, 124.243862, 0)
n3l0 = Node(8.239, 124.238, 0)

test_dir_configs = []

# valid directed edge:
test_dir_configs.append((n1l1, n2l1))

# invalid directed edge (no start node):
test_dir_configs.append((None, n2l1))

# invalid directed edge (no end node):
test_dir_configs.append((n1l1, None))

# invalid directed edge (same start and end node):
test_dir_configs.append((n1l1, n1l1))

# invalid layer combination:
test_dir_configs.append((n1l1, n2l0))

for config in test_dir_configs:
    start_node, end_node = config
    result = validate_call(
        f"DirEdge({start_node}, {end_node})",
        DirEdge,
        start_node,
        end_node
    )
    
# stitching test (only one of these connections is valid, the other should fail):
dir_edges_s = [DirEdge(n1l1, n2l1), DirEdge(n2l1, n3l1)]
dir_edges_e = [DirEdge(n1l2, n2l2), DirEdge(n2l1, n3l2)]
validate_call(
    "_stitch(dir_edges_s, dir_edges_e, weight=2, verbose=True)",
    _stitch,
    dir_edges_s,
    dir_edges_e,
    weight=2,
    verbose=True
)

# length test:
print("\nGreat-Circle Distance of MSU-IIT to 7-11 MSU-IIT:")
print(DirEdge(n1l1, n2l1).getLength())

# type test:
typeTestConfigs = [
    # start walk
    (n1l1, n2l1),
    # transfer walk
    (n1l3, n1l3),
    # invalid
    (n1l1, n2l0),
]
for config in typeTestConfigs:
    start_node, end_node = config
    dir_edge = validate_call(
        f"DirEdge({start_node}, {end_node})",
        DirEdge,
        start_node,
        end_node
    )
    if dir_edge is not None:
        print(f"Type of DirEdge({start_node}, {end_node}): {dir_edge.getType()}")

VALID: DirEdge(Node(N00181): lon=8.2393, lat=124.2384, layer=1, Node(N00182): lon=8.237768, lat=124.243862, layer=1)
INVALID: DirEdge(None, Node(N00182): lon=8.237768, lat=124.243862, layer=1) - [DIR EDGE] No start node provided.
INVALID: DirEdge(Node(N00181): lon=8.2393, lat=124.2384, layer=1, None) - [DIR EDGE] No end node provided.
INVALID: DirEdge(Node(N00181): lon=8.2393, lat=124.2384, layer=1, Node(N00181): lon=8.2393, lat=124.2384, layer=1) - [DIR EDGE] Start and end nodes cannot be identical.
INVALID: DirEdge(Node(N00181): lon=8.2393, lat=124.2384, layer=1, Node(N00191): lon=8.237768, lat=124.243862, layer=0) - [DIR EDGE] Invalid layer combination: 1 to 0
[DIR EDGE] Cannot connect N00181N00182 -> N00184N00185 (not connected)
[DIR EDGE] Connected N00181N00182 -> N00182N00186 with weight 2
[DIR EDGE] Cannot connect N00182N00183 -> N00184N00185 (not connected)
[DIR EDGE] Cannot connect N00182N00183 -> N00182N00186 (not connected)
Stitched 4 edges between sets of size 2 and 2
VALID

## **CityGraph** (in city_graph.py)